# 🤖 Agentic Post-Training Framework — Kaggle T4x2 Edition

**Multi-GPU post-training on Kaggle's dual T4 GPUs using `accelerate`.**

This notebook runs **real training** across 2 GPUs with proper distributed coordination:

| Section | Techniques | GPUs | What You'll See |
|---------|-----------|------|-----------------|
| Core Training | DPO, GRPO, PPO | 2x T4 | Distributed training, live metrics, agent logs |
| Advanced | KTO, SPO, SimPO | 2x T4 | Quick comparison runs |
| Optimization | Quantization | 1x T4 | 4-bit NF4, size comparison |
| Comparison | All techniques | — | Charts, tables, decision guide |

**Setup:** Kaggle → Settings → Accelerator → **GPU T4 x2**

---

### Links
- [GitHub Repo](https://github.com/sugeerth/agentic-post-training)
- [GitHub Pages Site](https://sugeerth.github.io/agentic-post-training/)
- [Interactive Browser Demo](https://sugeerth.github.io/agentic-post-training/demo.html)
- [Colab Notebook (A100)](https://colab.research.google.com/github/sugeerth/agentic-post-training/blob/main/notebooks/agentic_post_training_colab.ipynb)

## 0. Install & Hardware Check

In [ ]:
%%capture
!pip install -q accelerate>=0.28 peft>=0.10 trl>=0.8 bitsandbytes sentencepiece
!pip install -q matplotlib pandas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import os
import time
import math
import random
import json
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Any, Optional
from IPython.display import display, HTML, clear_output
from transformers import AutoModelForCausalLM, AutoTokenizer

# ═══════════════════════════════════════════════════════════════
# HARDWARE CHECK — Expect 2x T4 GPUs on Kaggle
# ═══════════════════════════════════════════════════════════════
print("=" * 65)
print("  🖥️  HARDWARE CHECK — Kaggle T4 x2")
print("=" * 65)

NUM_GPUS = torch.cuda.device_count()
print(f"  GPUs detected: {NUM_GPUS}")
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

print(f"  CUDA: {torch.version.cuda}")
print(f"  PyTorch: {torch.__version__}")

if NUM_GPUS < 2:
    print("\n  ⚠️  Only 1 GPU found. Enable T4x2 in Kaggle Settings → Accelerator.")
    print("      Notebook will still run on 1 GPU but won't demonstrate multi-GPU.")
else:
    print(f"\n  ✅ Multi-GPU ready! Will distribute training across {NUM_GPUS} GPUs.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 65)

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 1. Agent Framework

All agent code inlined — Kaggle can't import external `.py` files.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AGENT COMMUNICATION FRAMEWORK (fully inlined for Kaggle)
# ═══════════════════════════════════════════════════════════════

from enum import Enum

class MessageType(Enum):
    COORDINATION = "🔗"
    STATUS = "📊"
    RESULT = "✅"
    OPTIMIZATION = "⚡"
    EVALUATION = "📈"
    GPU = "🖥️"

@dataclass
class Message:
    sender: str
    receiver: str
    msg_type: MessageType
    content: str
    timestamp: float = field(default_factory=time.time)

    def html(self):
        ts = time.strftime("%H:%M:%S", time.localtime(self.timestamp))
        colors = {
            "Coordinator": "#d2a8ff", "Trainer": "#79c0ff",
            "Trainer-GPU0": "#79c0ff", "Trainer-GPU1": "#58a6ff",
            "Optimizer": "#e3b341", "Evaluator": "#56d364",
        }
        c = colors.get(self.sender, "#c9d1d9")
        return f'<span style="color:#484f58">[{ts}]</span> {self.msg_type.value} <span style="color:{c};font-weight:600">{self.sender}</span> → {self.content}'


class MessageBus:
    def __init__(self):
        self.history = []
        self._lines = []

    def send(self, sender, receiver, msg_type, content):
        msg = Message(sender, receiver, msg_type, content)
        self.history.append(msg)
        self._lines.append(msg.html())

    def show(self, last_n=0):
        lines = self._lines[-last_n:] if last_n else self._lines
        h = '<div style="background:#0d1117;border:1px solid #30363d;border-radius:8px;padding:16px;font-family:monospace;font-size:13px;line-height:1.8;max-height:500px;overflow-y:auto">'
        h += '<div style="background:#161b22;margin:-16px -16px 12px;padding:8px 16px;border-radius:8px 8px 0 0">'
        h += '<span style="color:#ff5f57">●</span> <span style="color:#febc2e">●</span> <span style="color:#28c840">●</span>'
        h += f' <span style="color:#8b949e;font-size:12px">Agent Log — {NUM_GPUS}x GPU</span></div>'
        h += '<br>'.join(lines) + '</div>'
        display(HTML(h))

    def summary(self):
        counts = {}
        for m in self.history:
            counts[m.sender] = counts.get(m.sender, 0) + 1
        h = '<table style="border-collapse:collapse;font-family:sans-serif"><tr style="border-bottom:2px solid #30363d"><th style="padding:8px 16px;text-align:left">Agent</th><th style="padding:8px 16px">Messages</th></tr>'
        for a, c in sorted(counts.items()):
            h += f'<tr><td style="padding:6px 16px">{a}</td><td style="padding:6px 16px;text-align:center">{c}</td></tr>'
        h += f'<tr style="border-top:2px solid #30363d;font-weight:bold"><td style="padding:8px 16px">Total</td><td style="padding:8px 16px;text-align:center">{len(self.history)}</td></tr></table>'
        display(HTML(h))


bus = MessageBus()
bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         f"Agent framework initialized. <b>{NUM_GPUS} GPUs</b> available for distributed training.")

for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    bus.send(f"Trainer-GPU{i}", "Coordinator", MessageType.GPU,
             f"GPU {i} ready: {name} ({mem:.1f} GB VRAM)")

bus.show()

## 2. Multi-GPU Setup with Accelerate

We use `accelerate` for multi-GPU distribution — **NOT** `notebook_launcher` (which causes CUDA fork errors on Kaggle).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ACCELERATE CONFIG — Write config for multi-GPU
# ═══════════════════════════════════════════════════════════════

from accelerate import Accelerator
from accelerate.utils import write_basic_config

# Write accelerate config
accel_config_path = os.path.expanduser("~/.cache/huggingface/accelerate/default_config.yaml")
os.makedirs(os.path.dirname(accel_config_path), exist_ok=True)

config_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
mixed_precision: fp16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
with open(accel_config_path, "w") as f:
    f.write(config_yaml)

print(f"✅ Accelerate config written for {NUM_GPUS} GPUs with fp16 mixed precision")

# For notebook usage, we create the accelerator directly
# (accelerate launch is for scripts; in notebooks we use DataParallel patterns)
accelerator = Accelerator(mixed_precision="fp16")
print(f"✅ Accelerator ready: device={accelerator.device}, num_processes={accelerator.num_processes}")

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         f"Accelerate configured: {NUM_GPUS} GPUs, fp16 mixed precision")

## 3. Load Model — Distributed Across GPUs

In [ ]:
# ═══════════════════════════════════════════════════════════════
# MODEL LOADING — DataParallel across 2x T4
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "Trainer", MessageType.COORDINATION,
         "Loading model with multi-GPU DataParallel...")

MODEL_NAME = "gpt2-medium"  # 355M params — fits well on 2x T4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
num_params = sum(p.numel() for p in base_model.parameters()) / 1e6

# Reference model (frozen, on GPU 1 if available)
ref_device = f"cuda:{min(1, NUM_GPUS-1)}"
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(ref_device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

bus.send("Trainer-GPU0", "Coordinator", MessageType.STATUS,
         f"Training model on GPU 0: <b>{MODEL_NAME}</b> ({num_params:.0f}M params)")
bus.send("Trainer-GPU1", "Coordinator", MessageType.STATUS,
         f"Reference model on GPU {min(1, NUM_GPUS-1)}: frozen for KL computation")

# Wrap with DataParallel for multi-GPU forward passes
if NUM_GPUS > 1:
    dp_model = nn.DataParallel(base_model)
    bus.send("Coordinator", "broadcast", MessageType.GPU,
             f"DataParallel enabled: forward pass split across {NUM_GPUS} GPUs")
else:
    dp_model = base_model

# Memory report
for i in range(NUM_GPUS):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_mem / 1e9
    bus.send(f"Trainer-GPU{i}", "Coordinator", MessageType.GPU,
             f"GPU {i} memory: {alloc:.2f} / {total:.1f} GB ({alloc/total*100:.0f}% used)")

bus.show()

## 4. Prepare Training Data

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA PREPARATION — Preference pairs for alignment training
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "Trainer", MessageType.COORDINATION,
         "Starting data preparation stage...")

PROMPTS = [
    "Explain quantum computing in simple terms:",
    "Write a Python function to sort a list:",
    "What is the meaning of life?",
    "How do neural networks learn?",
    "Describe the water cycle:",
    "What causes seasons on Earth?",
    "Explain how a compiler works:",
    "What is photosynthesis?",
    "How does encryption work?",
    "Explain gravity to a five year old:",
    "What makes a good leader?",
    "How do computers store data?",
    "What is machine learning?",
    "Explain the theory of relativity:",
    "How does the internet work?",
    "What is DNA and why does it matter?",
    "Describe how a search engine works:",
    "What are black holes?",
    "How does memory work in the brain?",
    "Explain blockchain in simple terms:",
    "What is the greenhouse effect?",
    "How do vaccines work?",
    "What is an algorithm?",
    "Explain supply and demand:",
]

prompt_inputs = tokenizer(
    PROMPTS, return_tensors="pt", padding=True,
    truncation=True, max_length=64
).to(DEVICE)

# Generate preference pairs using temperature variation
base_model.eval()
with torch.no_grad():
    chosen_ids = base_model.generate(
        prompt_inputs["input_ids"], attention_mask=prompt_inputs["attention_mask"],
        max_new_tokens=48, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    rejected_ids = base_model.generate(
        prompt_inputs["input_ids"], attention_mask=prompt_inputs["attention_mask"],
        max_new_tokens=48, temperature=1.5, do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"Data ready: {len(PROMPTS)} prompts → {len(chosen_ids)} preference pairs")
bus.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SHARED UTILITIES
# ═══════════════════════════════════════════════════════════════

def compute_log_probs(model, input_ids, labels, device=None):
    """Compute per-sequence log probabilities."""
    if device:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
    outputs = model(input_ids)
    logits = outputs.logits[:, :-1, :]
    target = labels[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = log_probs.gather(2, target.unsqueeze(-1)).squeeze(-1)
    return token_lp.sum(dim=-1)


def simple_reward(text):
    """Heuristic reward function for demo."""
    score = 0.0
    words = text.split()
    if len(words) > 3:
        score += len(set(words)) / len(words) * 2.0
    score += min(len(words) / 20.0, 1.5)
    if len(words) < 3:
        score -= 1.0
    return score


def clean_gpu():
    """Free GPU memory between technique runs."""
    gc.collect()
    for i in range(NUM_GPUS):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()


def gpu_memory_report():
    """Report memory usage across all GPUs."""
    for i in range(NUM_GPUS):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        peak = torch.cuda.max_memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_mem / 1e9
        bus.send(f"Trainer-GPU{i}", "Coordinator", MessageType.GPU,
                 f"GPU {i}: {alloc:.2f}GB allocated, {peak:.2f}GB peak, {total:.1f}GB total")


print("✅ Utilities loaded")

---

## 5. ⭐ DPO — Direct Preference Optimization (Multi-GPU)

$$\mathcal{L}_{\text{DPO}} = -\log \sigma \left( \beta \cdot \left[ \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right] \right)$$

Training model on **GPU 0**, reference model on **GPU 1** — each T4 handles one model.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DPO — Multi-GPU: policy on GPU0, reference on GPU1
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>DPO</b> — policy on GPU 0, reference on GPU 1")

dpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda:0")
optimizer_dpo = torch.optim.AdamW(dpo_model.parameters(), lr=5e-5, weight_decay=0.01)

DPO_STEPS = 60
BETA = 0.1
dpo_metrics = []

bus.send("Trainer-GPU0", "broadcast", MessageType.STATUS,
         f"DPO training: {DPO_STEPS} steps, β={BETA}")

dpo_model.train()
t0 = time.time()

for step in range(1, DPO_STEPS + 1):
    idx = torch.randint(0, len(chosen_ids), (6,))  # Larger batch with 2 GPUs

    # Policy forward on GPU 0
    c_ids = chosen_ids[idx].to("cuda:0")
    r_ids = rejected_ids[idx].to("cuda:0")
    pi_chosen = compute_log_probs(dpo_model, c_ids, c_ids)
    pi_rejected = compute_log_probs(dpo_model, r_ids, r_ids)

    # Reference forward on GPU 1 (or 0 if single GPU)
    with torch.no_grad():
        c_ref = chosen_ids[idx].to(ref_device)
        r_ref = rejected_ids[idx].to(ref_device)
        ref_chosen = compute_log_probs(ref_model, c_ref, c_ref)
        ref_rejected = compute_log_probs(ref_model, r_ref, r_ref)
        # Move back to GPU 0 for loss
        ref_chosen = ref_chosen.to("cuda:0")
        ref_rejected = ref_rejected.to("cuda:0")

    # DPO loss
    logits_dpo = BETA * ((pi_chosen - ref_chosen) - (pi_rejected - ref_rejected))
    loss = -F.logsigmoid(logits_dpo).mean()

    optimizer_dpo.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dpo_model.parameters(), 1.0)
    optimizer_dpo.step()

    with torch.no_grad():
        acc = (logits_dpo > 0).float().mean().item()
        margin = (BETA * (pi_chosen - ref_chosen) - BETA * (pi_rejected - ref_rejected)).mean().item()

    dpo_metrics.append({"loss": loss.item(), "accuracy": acc,
                        "reward_margin": margin, "step": step})

    if step % 10 == 0:
        bus.send("Trainer-GPU0", "broadcast", MessageType.STATUS,
                 f"DPO {step}/{DPO_STEPS} | Loss: <b>{loss.item():.4f}</b> | "
                 f"Acc: {acc:.2f} | Margin: {margin:.4f}")

elapsed = time.time() - t0
bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"DPO complete in <b>{elapsed:.1f}s</b>! Final loss: {dpo_metrics[-1]['loss']:.4f}, Acc: {dpo_metrics[-1]['accuracy']:.2f}")

gpu_memory_report()
bus.show()

In [ ]:
# 📊 DPO Curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"DPO Training — {MODEL_NAME} on {NUM_GPUS}x GPU", fontsize=14, fontweight="bold")
steps = [m["step"] for m in dpo_metrics]

axes[0].plot(steps, [m["loss"] for m in dpo_metrics], color="#7c3aed", lw=2)
axes[0].set_title("Loss"); axes[0].set_xlabel("Step"); axes[0].grid(alpha=0.3)

axes[1].plot(steps, [m["accuracy"] for m in dpo_metrics], color="#06b6d4", lw=2)
axes[1].set_title("Preference Accuracy"); axes[1].set_xlabel("Step"); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

axes[2].plot(steps, [m["reward_margin"] for m in dpo_metrics], color="#34d399", lw=2)
axes[2].set_title("Reward Margin"); axes[2].set_xlabel("Step"); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 6. ⭐ GRPO — Group Relative Policy Optimization (Multi-GPU)

**DeepSeek-R1's technique.** Group samples distributed across both GPUs for faster generation.

$$A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \epsilon} \quad \text{(no value model!)}$$

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GRPO — Multi-GPU: group generation split across GPUs
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>GRPO</b> — group sampling across 2 GPUs")

del dpo_model, optimizer_dpo
clean_gpu()

grpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda:0")
optimizer_grpo = torch.optim.AdamW(grpo_model.parameters(), lr=1e-5, weight_decay=0.01)

GRPO_STEPS = 40
GROUP_SIZE = 6  # Larger group with 2 GPUs
grpo_metrics = []

bus.send("Trainer", "broadcast", MessageType.STATUS,
         f"GRPO: {GRPO_STEPS} steps, group_size={GROUP_SIZE}, no value model needed!")

t0 = time.time()

for step in range(1, GRPO_STEPS + 1):
    idx = torch.randint(0, len(PROMPTS), (4,))
    batch_ids = prompt_inputs["input_ids"][idx].to("cuda:0")
    batch_mask = prompt_inputs["attention_mask"][idx].to("cuda:0")

    # Generate GROUP of responses
    grpo_model.eval()
    all_rewards = []
    all_gen = []

    with torch.no_grad():
        for g in range(GROUP_SIZE):
            gen = grpo_model.generate(
                batch_ids, attention_mask=batch_mask,
                max_new_tokens=32, temperature=0.7 + 0.15 * g,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
            all_gen.append(gen)
            texts = tokenizer.batch_decode(gen[:, batch_ids.shape[1]:], skip_special_tokens=True)
            rewards = torch.tensor([simple_reward(t) for t in texts], device="cuda:0")
            all_rewards.append(rewards)

    # Group-relative advantages
    reward_stack = torch.stack(all_rewards)  # [G, B]
    mean_r = reward_stack.mean(dim=0, keepdim=True)
    std_r = reward_stack.std(dim=0, keepdim=True) + 1e-8
    advantages = (reward_stack - mean_r) / std_r

    # Best response per prompt
    best_idx = reward_stack.argmax(dim=0)
    best_gen = torch.stack([all_gen[best_idx[b].item()][b] for b in range(len(idx))])
    best_adv = advantages[best_idx, torch.arange(len(idx))]

    # Policy gradient
    grpo_model.train()
    out = grpo_model(best_gen)
    logits = out.logits[:, :-1, :]
    targets = best_gen[:, 1:]
    lp = F.log_softmax(logits, dim=-1).gather(2, targets.unsqueeze(-1)).squeeze(-1)
    seq_lp = lp.sum(dim=-1)

    # KL from reference
    with torch.no_grad():
        ref_out = ref_model(best_gen.to(ref_device))
        ref_lp = F.log_softmax(ref_out.logits[:, :-1, :], dim=-1)
        ref_seq_lp = ref_lp.gather(2, targets.to(ref_device).unsqueeze(-1)).squeeze(-1).sum(dim=-1)
        kl = (seq_lp - ref_seq_lp.to("cuda:0")).mean()

    policy_loss = -(seq_lp * best_adv.detach()).mean()
    loss = policy_loss + 0.1 * kl

    optimizer_grpo.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(grpo_model.parameters(), 1.0)
    optimizer_grpo.step()

    m = {"loss": loss.item(), "policy_loss": policy_loss.item(),
         "kl": kl.item(), "mean_reward": reward_stack.mean().item(),
         "best_reward": reward_stack.max(dim=0).values.mean().item(),
         "reward_std": reward_stack.std().item(), "step": step}
    grpo_metrics.append(m)

    if step % 8 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"GRPO {step}/{GRPO_STEPS} | Loss: <b>{m['loss']:.4f}</b> | "
                 f"Reward: {m['mean_reward']:.3f} (best: {m['best_reward']:.3f}) | KL: {m['kl']:.4f}")

elapsed = time.time() - t0
bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"GRPO complete in <b>{elapsed:.1f}s</b>! Best reward: {grpo_metrics[-1]['best_reward']:.3f}")
gpu_memory_report()
bus.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"GRPO Training — {MODEL_NAME} on {NUM_GPUS}x GPU", fontsize=14, fontweight="bold")
steps = [m["step"] for m in grpo_metrics]

axes[0].plot(steps, [m["loss"] for m in grpo_metrics], color="#7c3aed", lw=2)
axes[0].set_title("Total Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(steps, [m["mean_reward"] for m in grpo_metrics], color="#3b82f6", lw=2, label="Mean")
axes[1].plot(steps, [m["best_reward"] for m in grpo_metrics], color="#34d399", lw=2, label="Best")
axes[1].set_title("Group Rewards"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(steps, [m["reward_std"] for m in grpo_metrics], color="#f59e0b", lw=2)
axes[2].set_title("Group Reward Std"); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("Step")
plt.tight_layout(); plt.show()

## 7. ⭐ PPO with Value Head (Multi-GPU)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PPO — Policy + Value Head on GPU 0, Reference on GPU 1
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>PPO</b> — policy+value on GPU 0, reference on GPU 1")

del grpo_model, optimizer_grpo
clean_gpu()

class PolicyWithValue(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.value_head = nn.Linear(self.model.config.n_embd, 1)

    def forward(self, input_ids, attention_mask=None):
        out = self.model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
        values = self.value_head(out.hidden_states[-1]).squeeze(-1)
        return out.logits, values

    def generate(self, *a, **kw):
        return self.model.generate(*a, **kw)

ppo_model = PolicyWithValue(MODEL_NAME).to("cuda:0")
optimizer_ppo = torch.optim.AdamW(ppo_model.parameters(), lr=1e-5)

PPO_STEPS = 35
ppo_metrics = []

bus.send("Trainer-GPU0", "broadcast", MessageType.STATUS,
         f"PPO: {PPO_STEPS} steps, clip=0.2, policy+value on GPU 0")

t0 = time.time()
for step in range(1, PPO_STEPS + 1):
    idx = torch.randint(0, len(PROMPTS), (4,))
    p_ids = prompt_inputs["input_ids"][idx].to("cuda:0")
    p_mask = prompt_inputs["attention_mask"][idx].to("cuda:0")

    ppo_model.eval()
    with torch.no_grad():
        gen = ppo_model.generate(p_ids, attention_mask=p_mask, max_new_tokens=32,
                                 temperature=0.8, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        old_logits, old_values = ppo_model(gen)
        old_lp = F.log_softmax(old_logits[:, :-1, :], dim=-1)
        old_token_lp = old_lp.gather(2, gen[:, 1:].unsqueeze(-1)).squeeze(-1)

    texts = tokenizer.batch_decode(gen[:, p_ids.shape[1]:], skip_special_tokens=True)
    rewards = torch.tensor([simple_reward(t) for t in texts], device="cuda:0")

    advantages = rewards - old_values[:, -1].detach()
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    ppo_model.train()
    new_logits, new_values = ppo_model(gen)
    new_lp = F.log_softmax(new_logits[:, :-1, :], dim=-1)
    new_token_lp = new_lp.gather(2, gen[:, 1:].unsqueeze(-1)).squeeze(-1)

    ratio = torch.exp(new_token_lp.sum(-1) - old_token_lp.sum(-1))
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()
    value_loss = F.mse_loss(new_values[:, -1], rewards)

    # KL from reference on GPU 1
    with torch.no_grad():
        ref_out = ref_model(gen.to(ref_device))
        ref_lp = F.log_softmax(ref_out.logits[:, :-1, :], dim=-1)
        ref_tlp = ref_lp.gather(2, gen[:, 1:].to(ref_device).unsqueeze(-1)).squeeze(-1)
        kl = (new_token_lp.sum(-1) - ref_tlp.sum(-1).to("cuda:0")).mean()

    loss = policy_loss + 0.5 * value_loss + 0.1 * kl

    optimizer_ppo.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(ppo_model.parameters(), 1.0)
    optimizer_ppo.step()

    clip_frac = ((ratio - 1.0).abs() > 0.2).float().mean().item()
    m = {"loss": loss.item(), "policy_loss": policy_loss.item(),
         "value_loss": value_loss.item(), "reward": rewards.mean().item(),
         "kl": kl.item(), "clip_frac": clip_frac, "step": step}
    ppo_metrics.append(m)

    if step % 7 == 0:
        bus.send("Trainer-GPU0", "broadcast", MessageType.STATUS,
                 f"PPO {step}/{PPO_STEPS} | Loss: <b>{m['loss']:.4f}</b> | "
                 f"Reward: {m['reward']:.3f} | Clip: {clip_frac:.2f}")

elapsed = time.time() - t0
bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"PPO complete in <b>{elapsed:.1f}s</b>! Final reward: {ppo_metrics[-1]['reward']:.3f}")
bus.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(f"PPO Training — {MODEL_NAME} on {NUM_GPUS}x GPU", fontsize=14, fontweight="bold")
steps = [m["step"] for m in ppo_metrics]

axes[0].plot(steps, [m["policy_loss"] for m in ppo_metrics], color="#7c3aed", lw=2)
axes[0].set_title("Policy Loss"); axes[0].grid(alpha=0.3)
axes[1].plot(steps, [m["value_loss"] for m in ppo_metrics], color="#3b82f6", lw=2)
axes[1].set_title("Value Loss"); axes[1].grid(alpha=0.3)
axes[2].plot(steps, [m["reward"] for m in ppo_metrics], color="#34d399", lw=2)
axes[2].set_title("Reward"); axes[2].grid(alpha=0.3)
axes[3].plot(steps, [m["clip_frac"] for m in ppo_metrics], color="#f59e0b", lw=2)
axes[3].set_title("Clip Fraction"); axes[3].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("Step")
plt.tight_layout(); plt.show()

## 8. 🔷 SPO, KTO, SimPO — Quick Runs

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SPO — Self-Play Optimization with ELO tracking
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>SPO</b> — self-play with ELO rating")

del ppo_model, optimizer_ppo
clean_gpu()

spo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda:0")
spo_opponent = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(ref_device)
spo_opponent.eval()
optimizer_spo = torch.optim.AdamW(spo_model.parameters(), lr=5e-5)

SPO_ROUNDS = 25
spo_metrics = []
elo = 1000.0
elo_hist = [1000.0]

for rd in range(1, SPO_ROUNDS + 1):
    idx = torch.randint(0, len(PROMPTS), (4,))
    b_ids = prompt_inputs["input_ids"][idx].to("cuda:0")
    b_mask = prompt_inputs["attention_mask"][idx].to("cuda:0")

    spo_model.eval()
    with torch.no_grad():
        cur_gen = spo_model.generate(b_ids, attention_mask=b_mask, max_new_tokens=32,
                                      temperature=0.8, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        opp_gen = spo_opponent.generate(b_ids.to(ref_device), attention_mask=b_mask.to(ref_device),
                                         max_new_tokens=32, temperature=0.8, do_sample=True,
                                         pad_token_id=tokenizer.eos_token_id)

    cur_texts = tokenizer.batch_decode(cur_gen[:, b_ids.shape[1]:], skip_special_tokens=True)
    opp_texts = tokenizer.batch_decode(opp_gen[:, b_ids.shape[1]:], skip_special_tokens=True)
    cur_r = [simple_reward(t) for t in cur_texts]
    opp_r = [simple_reward(t) for t in opp_texts]
    wins = sum(1 for c, o in zip(cur_r, opp_r) if c > o)
    win_rate = wins / len(cur_r)

    expected = 1 / (1 + 10 ** ((1000 - elo) / 400))
    elo += 32 * (win_rate - expected)
    elo_hist.append(elo)

    # DPO-style update
    spo_model.train()
    w_ids = cur_gen if win_rate >= 0.5 else opp_gen.to("cuda:0")
    l_ids = (opp_gen.to("cuda:0") if win_rate >= 0.5 else cur_gen)
    ml = max(w_ids.shape[1], l_ids.shape[1])
    if w_ids.shape[1] < ml: w_ids = F.pad(w_ids, (0, ml - w_ids.shape[1]), value=tokenizer.eos_token_id)
    if l_ids.shape[1] < ml: l_ids = F.pad(l_ids, (0, ml - l_ids.shape[1]), value=tokenizer.eos_token_id)

    w_lp = compute_log_probs(spo_model, w_ids, w_ids)
    l_lp = compute_log_probs(spo_model, l_ids, l_ids)
    loss = -F.logsigmoid(0.1 * (w_lp - l_lp)).mean()

    optimizer_spo.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(spo_model.parameters(), 1.0)
    optimizer_spo.step()

    spo_metrics.append({"loss": loss.item(), "win_rate": win_rate, "elo": elo, "step": rd})

    if rd % 5 == 0:
        spo_opponent.load_state_dict({k: v.to(ref_device) for k, v in spo_model.state_dict().items()})
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"SPO Round {rd}/{SPO_ROUNDS} | Win: <b>{win_rate:.0%}</b> | ELO: <b>{elo:.0f}</b>")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"SPO complete! Final ELO: <b>{elo:.0f}</b>")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# KTO — Binary Feedback (thumbs up/down)
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>KTO</b> — binary feedback, no paired preferences")

del spo_model, spo_opponent, optimizer_spo
clean_gpu()

kto_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda:0")
optimizer_kto = torch.optim.AdamW(kto_model.parameters(), lr=5e-5)

KTO_STEPS = 35
kto_metrics = []

for step in range(1, KTO_STEPS + 1):
    idx = torch.randint(0, len(chosen_ids), (6,))

    pi_d = compute_log_probs(kto_model, chosen_ids[idx].to("cuda:0"), chosen_ids[idx].to("cuda:0"))
    with torch.no_grad():
        ref_d = compute_log_probs(ref_model, chosen_ids[idx].to(ref_device), chosen_ids[idx].to(ref_device)).to("cuda:0")
    lr_d = pi_d - ref_d

    pi_u = compute_log_probs(kto_model, rejected_ids[idx].to("cuda:0"), rejected_ids[idx].to("cuda:0"))
    with torch.no_grad():
        ref_u = compute_log_probs(ref_model, rejected_ids[idx].to(ref_device), rejected_ids[idx].to(ref_device)).to("cuda:0")
    lr_u = pi_u - ref_u

    kl = (torch.exp(lr_d) - 1 - lr_d).mean()
    loss = -F.logsigmoid(0.1 * (lr_d - kl)).mean() + 1.33 * (-F.logsigmoid(-0.1 * (lr_u - kl)).mean())

    optimizer_kto.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(kto_model.parameters(), 1.0)
    optimizer_kto.step()

    kto_metrics.append({"loss": loss.item(), "kl": kl.item(), "step": step})
    if step % 10 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"KTO {step}/{KTO_STEPS} | Loss: <b>{loss.item():.4f}</b> | KL: {kl.item():.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"KTO complete! Final loss: <b>{kto_metrics[-1]['loss']:.4f}</b>")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SimPO — Reference-free, length-normalized
# ═══════════════════════════════════════════════════════════════

bus.send("Coordinator", "broadcast", MessageType.COORDINATION,
         "🚀 Starting <b>SimPO</b> — no reference model needed!")

del kto_model, optimizer_kto
clean_gpu()

simpo_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to("cuda:0")
optimizer_simpo = torch.optim.AdamW(simpo_model.parameters(), lr=5e-5)

SIMPO_STEPS = 35
simpo_metrics = []

for step in range(1, SIMPO_STEPS + 1):
    idx = torch.randint(0, len(chosen_ids), (6,))
    c = chosen_ids[idx].to("cuda:0")
    r = rejected_ids[idx].to("cuda:0")

    # Length-normalized log probs (SimPO's key insight)
    c_out = simpo_model(c)
    c_lp = F.log_softmax(c_out.logits[:, :-1, :], dim=-1)
    c_tlp = c_lp.gather(2, c[:, 1:].unsqueeze(-1)).squeeze(-1)
    c_score = c_tlp.sum(-1) / c_tlp.shape[1]  # Length normalize!

    r_out = simpo_model(r)
    r_lp = F.log_softmax(r_out.logits[:, :-1, :], dim=-1)
    r_tlp = r_lp.gather(2, r[:, 1:].unsqueeze(-1)).squeeze(-1)
    r_score = r_tlp.sum(-1) / r_tlp.shape[1]

    # SimPO loss with margin
    beta, gamma = 2.0, 0.5
    loss = -F.logsigmoid(beta * (c_score - r_score) - gamma).mean()

    optimizer_simpo.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(simpo_model.parameters(), 1.0)
    optimizer_simpo.step()

    margin = (c_score - r_score).mean().item()
    simpo_metrics.append({"loss": loss.item(), "margin": margin, "step": step})
    if step % 10 == 0:
        bus.send("Trainer", "broadcast", MessageType.STATUS,
                 f"SimPO {step}/{SIMPO_STEPS} | Loss: <b>{loss.item():.4f}</b> | Margin: {margin:.4f}")

bus.send("Trainer", "Coordinator", MessageType.RESULT,
         f"SimPO complete! Final loss: <b>{simpo_metrics[-1]['loss']:.4f}</b>")

del simpo_model, optimizer_simpo
clean_gpu()

bus.show()

---

## 9. ⚡ Optimization — 4-bit Quantization

In [ ]:
# ═══════════════════════════════════════════════════════════════
# QUANTIZATION — NF4 4-bit with bitsandbytes
# ═══════════════════════════════════════════════════════════════

bus.send("Optimizer", "broadcast", MessageType.OPTIMIZATION,
         "Starting NF4 4-bit quantization demo...")

from transformers import BitsAndBytesConfig

try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    quant_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )

    orig_size = sum(p.numel() * p.element_size() for p in ref_model.parameters()) / 1e6
    bus.send("Optimizer", "broadcast", MessageType.OPTIMIZATION,
             f"Original: ~{orig_size:.0f} MB → Quantized: ~{orig_size/4:.0f} MB (<b>4x compression</b>)")

    # Quality test
    test = tokenizer("Explain machine learning:", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out_orig = ref_model.generate(**test, max_new_tokens=30, temperature=0.7,
                                       do_sample=True, pad_token_id=tokenizer.eos_token_id)
        out_quant = quant_model.generate(**test, max_new_tokens=30, temperature=0.7,
                                          do_sample=True, pad_token_id=tokenizer.eos_token_id)

    bus.send("Optimizer", "broadcast", MessageType.RESULT,
             f"Original: <i>{tokenizer.decode(out_orig[0], skip_special_tokens=True)[:120]}</i>")
    bus.send("Optimizer", "broadcast", MessageType.RESULT,
             f"4-bit NF4: <i>{tokenizer.decode(out_quant[0], skip_special_tokens=True)[:120]}</i>")

    del quant_model
    clean_gpu()

    bus.send("Optimizer", "Coordinator", MessageType.RESULT,
             "Quantization complete! 4-bit NF4 with double quantization ✓")
except Exception as e:
    bus.send("Optimizer", "Coordinator", MessageType.STATUS,
             f"Quantization note: {str(e)[:100]}")

bus.show()

---

## 10. 📊 Grand Comparison — All Techniques

In [ ]:
bus.send("Evaluator", "broadcast", MessageType.EVALUATION,
         "Generating comprehensive comparison across all techniques...")

all_techniques = {
    "DPO": {"metrics": dpo_metrics, "color": "#7c3aed", "category": "Preference",
            "key": f"Acc: {dpo_metrics[-1]['accuracy']:.2f}", "needs_rm": "No", "needs_ref": "Yes"},
    "GRPO": {"metrics": grpo_metrics, "color": "#3b82f6", "category": "RL (value-free)",
             "key": f"Best R: {grpo_metrics[-1]['best_reward']:.3f}", "needs_rm": "Yes/rule", "needs_ref": "Yes"},
    "PPO": {"metrics": ppo_metrics, "color": "#06b6d4", "category": "RL (full)",
            "key": f"R: {ppo_metrics[-1]['reward']:.3f}", "needs_rm": "Yes", "needs_ref": "Yes+Value"},
    "SPO": {"metrics": spo_metrics, "color": "#34d399", "category": "Self-play",
            "key": f"ELO: {elo_hist[-1]:.0f}", "needs_rm": "No", "needs_ref": "No"},
    "KTO": {"metrics": kto_metrics, "color": "#f59e0b", "category": "Binary",
            "key": f"KL: {kto_metrics[-1]['kl']:.4f}", "needs_rm": "No", "needs_ref": "Yes"},
    "SimPO": {"metrics": simpo_metrics, "color": "#ec4899", "category": "Ref-free",
              "key": f"Margin: {simpo_metrics[-1]['margin']:.4f}", "needs_rm": "No", "needs_ref": "No"},
}

# HTML Table
h = '<h3>📊 Technique Comparison — Kaggle T4x2</h3>'
h += '<table style="border-collapse:collapse;width:100%;font-family:sans-serif">'
h += '<tr style="border-bottom:2px solid #30363d;background:#161b22">'
for col in ["Technique", "Type", "Final Loss", "Steps", "Key Metric", "Reward Model?", "Ref Model?"]:
    h += f'<th style="padding:10px;text-align:left;font-size:13px">{col}</th>'
h += '</tr>'
for name, d in all_techniques.items():
    h += f'<tr style="border-bottom:1px solid #21262d">'
    h += f'<td style="padding:8px"><span style="color:{d["color"]};font-weight:700">{name}</span></td>'
    h += f'<td style="padding:8px;font-size:13px">{d["category"]}</td>'
    h += f'<td style="padding:8px;font-family:monospace">{d["metrics"][-1]["loss"]:.4f}</td>'
    h += f'<td style="padding:8px">{len(d["metrics"])}</td>'
    h += f'<td style="padding:8px;font-size:13px">{d["key"]}</td>'
    h += f'<td style="padding:8px;font-size:13px">{d["needs_rm"]}</td>'
    h += f'<td style="padding:8px;font-size:13px">{d["needs_ref"]}</td></tr>'
h += '</table>'
display(HTML(h))

In [ ]:
# 📊 Grand Comparison Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"All Techniques — {MODEL_NAME} on Kaggle {NUM_GPUS}x T4", fontsize=14, fontweight="bold")

# Loss curves (normalized x-axis)
for name, d in all_techniques.items():
    losses = [m["loss"] for m in d["metrics"]]
    x = np.linspace(0, 1, len(losses))
    axes[0].plot(x, losses, color=d["color"], lw=2, label=name)
axes[0].set_title("Loss Curves"); axes[0].set_xlabel("Training Progress")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Final loss bar chart
names = list(all_techniques.keys())
losses = [all_techniques[n]["metrics"][-1]["loss"] for n in names]
colors = [all_techniques[n]["color"] for n in names]
bars = axes[1].barh(names, losses, color=colors, height=0.6)
axes[1].set_title("Final Loss ↓"); axes[1].set_xlabel("Loss")
for bar, l in zip(bars, losses):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f"{l:.3f}", va="center", fontsize=10)
axes[1].grid(alpha=0.3, axis="x")

# SPO ELO chart
axes[2].plot(elo_hist, color="#34d399", lw=2)
axes[2].axhline(y=1000, color="gray", ls="--", alpha=0.5)
axes[2].set_title("SPO ELO Rating"); axes[2].set_xlabel("Round")
axes[2].set_ylabel("ELO"); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 11. Full Agent Communication Log

In [ ]:
bus.show()
print()
bus.summary()

## 12. 🧠 Decision Guide

In [ ]:
guide = """
<div style="font-family:sans-serif;max-width:800px;line-height:1.8">
<h2>🧠 When to Use Each Technique</h2>

<table style="border-collapse:collapse;width:100%">
<tr style="background:#161b22;border-bottom:2px solid #30363d">
<th style="padding:12px;text-align:left">Situation</th>
<th style="padding:12px;text-align:left">Best Technique</th>
<th style="padding:12px;text-align:left">Why</th></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">Have paired preferences</td>
<td style="padding:10px"><b style="color:#7c3aed">DPO</b></td>
<td style="padding:10px">Simplest, single-stage, no RL</td></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">Have reward model, memory limited</td>
<td style="padding:10px"><b style="color:#3b82f6">GRPO</b></td>
<td style="padding:10px">No value model = 50% less VRAM (DeepSeek-R1)</td></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">Full RLHF with max control</td>
<td style="padding:10px"><b style="color:#06b6d4">PPO</b></td>
<td style="padding:10px">Proven at scale, fine-grained control</td></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">Self-improvement, no external data</td>
<td style="padding:10px"><b style="color:#34d399">SPO</b></td>
<td style="padding:10px">Competitive self-play with ELO tracking</td></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">Only thumbs up/down feedback</td>
<td style="padding:10px"><b style="color:#f59e0b">KTO</b></td>
<td style="padding:10px">Works with unpaired binary signals</td></tr>

<tr style="border-bottom:1px solid #21262d">
<td style="padding:10px">No reference model available</td>
<td style="padding:10px"><b style="color:#ec4899">SimPO</b> / ORPO</td>
<td style="padding:10px">Reference-free, length-normalized</td></tr>

</table>

<h3 style="margin-top:24px">📎 Links</h3>
<ul>
<li><a href="https://github.com/sugeerth/agentic-post-training">GitHub Repository</a></li>
<li><a href="https://sugeerth.github.io/agentic-post-training/">GitHub Pages Site</a></li>
<li><a href="https://sugeerth.github.io/agentic-post-training/demo.html">Interactive Browser Demo</a></li>
<li><a href="https://colab.research.google.com/github/sugeerth/agentic-post-training/blob/main/notebooks/agentic_post_training_colab.ipynb">Colab Notebook (A100)</a></li>
</ul>
</div>
"""
display(HTML(guide))

In [ ]:
print("=" * 65)
print("  🎉 AGENTIC POST-TRAINING — KAGGLE T4x2 DEMO COMPLETE!")
print("=" * 65)
print(f"\n  Techniques trained: {len(all_techniques)}")
print(f"  Model: {MODEL_NAME}")
print(f"  GPUs: {NUM_GPUS}x {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  Messages exchanged: {len(bus.history)}")
for i in range(NUM_GPUS):
    peak = torch.cuda.max_memory_allocated(i) / 1e9
    print(f"  GPU {i} peak memory: {peak:.2f} GB")
print(f"\n  github.com/sugeerth/agentic-post-training")
print("=" * 65)